# Experimento de optimización de hiperparámetros — GPU (CUDA)

Este notebook implementa el flujo experimental completo para el dataset HTML Phishing (Trap4Phish 2025):
- **20,000 muestras**, 13 variables numéricas, clasificación binaria (0/1)
- **4 configuraciones de optimización** con Optuna:
  - MLP + TPE 
  - MLP + NSGA-II 
  - Residual MLP + TPE
  - Residual MLP + NSGA-II
  - **multi-objective:** F1 ↑, tiempo inferencia ↓, tamaño ↓
- **Dispositivo fijo:** `GPU (CUDA)` — todas las operaciones se ejecutan en `gpu`
- **Métricas registradas por trial:** F1-score, Accuracy, Tiempo de entrenamiento, Tiempo de inferencia, Tamaño del modelo (MB)

### Flujo general

```
Carga CSVs → Split 40% tuning → 4 estudios Optuna → Selección mejor trial/Pareto
    → Reentrenamiento completo → Guardado de 4 modelos → Evaluación en Test → Comparación
```

### Prerequisito
`EDA.ipynb` debe haberse ejecutado al menos una vez para generar los splits en `dataset/processed/`.  
Los modelos se guardarán en `models/gpu/`.

## 1. Dependencias y configuración global

**Decision:** Importar todas las dependencias al inicio y definir constantes experimentales en una sola celda de configuración.

**Why:** Centralizar la configuración permite ajustar el número de trials, épocas o rutas sin buscar en el notebook.

**Expected:** Entorno listo; sin errores de importación.

In [7]:
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import optuna
from optuna.samplers import TPESampler, NSGAIISampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

#warnings.filterwarnings("ignore")
print("Imports OK")

c:\Users\Mishell\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [8]:
# ── Dispositivo ──────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda")
print(f"Device: {DEVICE}")

# ── Hiperparámetros de Optuna ────────────────────────────────────────────────
N_TRIALS      = 30   # número de trials por estudio (ajustable)
TUNING_EPOCHS = 10   # épocas por trial durante la búsqueda
FINAL_EPOCHS  = 20   # épocas en el reentrenamiento final

# ── Arquitectura fija ────────────────────────────────────────────────────────
INPUT_DIM    = 13
FIXED_HIDDEN = 128

# ── Reproducibilidad ────────────────────────────────────────────────────────
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ── Rutas ────────────────────────────────────────────────────────────────────
DATA_DIR    = Path("../dataset/processed")
MODELS_DIR  = Path("models/gpu")
RESULTS_DIR = Path("results")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Models → {MODELS_DIR}")
print(f"Results → {RESULTS_DIR}")

Device: cuda
Models → models\gpu
Results → results


## 2. Carga de datos

**Decision:** Cargar directamente los splits procesados (ya escalados con StandardScaler) generados por `EDA.ipynb`.

**Why:** Evita re-escalar y garantiza consistencia con los experimentos del EDA. El notebook es autocontenido respecto a la etapa de optimización.

**Expected:** Train ~11,997 filas, Validation ~4,000, Test ~4,000; 13 columnas de features + columna `label`.

In [9]:
train_df = pd.read_csv(DATA_DIR / "HTML_train_scaled.csv")
val_df   = pd.read_csv(DATA_DIR / "HTML_val_scaled.csv")
test_df  = pd.read_csv(DATA_DIR / "HTML_test_scaled.csv")

FEATURE_COLS = [c for c in train_df.columns if c != "label"]

X_train = train_df[FEATURE_COLS].values
y_train = train_df["label"].values

X_val = val_df[FEATURE_COLS].values
y_val = val_df["label"].values

X_test = test_df[FEATURE_COLS].values
y_test = test_df["label"].values

print(f"Train:      {X_train.shape}  |  label balance: {y_train.mean():.3f}")
print(f"Validation: {X_val.shape}  |  label balance: {y_val.mean():.3f}")
print(f"Test:       {X_test.shape}  |  label balance: {y_test.mean():.3f}")
assert X_train.shape[1] == INPUT_DIM, f"Expected {INPUT_DIM} features, got {X_train.shape[1]}" 

Train:      (11997, 13)  |  label balance: 0.500
Validation: (4000, 13)  |  label balance: 0.500
Test:       (4000, 13)  |  label balance: 0.500


### 2.1 Conversión a tensores PyTorch

**Decision:** Convertir los arrays NumPy a tensores `float32` y moverlos al dispositivo configurado.

**Why:** PyTorch trabaja en `float32` por defecto; mover al dispositivo antes del entrenamiento evita transferencias repetidas en cada batch.

**Expected:** Tensores en el dispositivo correcto con shape `(N, 13)` para X y `(N, 1)` para y.

In [10]:
def to_tensor(X, y, device):
    xt = torch.tensor(X, dtype=torch.float32).to(device)
    yt = torch.tensor(y, dtype=torch.float32).view(-1, 1).to(device)
    return xt, yt

X_train_t, y_train_t = to_tensor(X_train, y_train, DEVICE)
X_val_t,   y_val_t   = to_tensor(X_val,   y_val,   DEVICE)
X_test_t,  y_test_t  = to_tensor(X_test,  y_test,  DEVICE)

print(f"X_train_t: {X_train_t.shape} on {X_train_t.device}")
print(f"X_val_t:   {X_val_t.shape}   on {X_val_t.device}")
print(f"X_test_t:  {X_test_t.shape}  on {X_test_t.device}")

X_train_t: torch.Size([11997, 13]) on cuda:0
X_val_t:   torch.Size([4000, 13])   on cuda:0
X_test_t:  torch.Size([4000, 13])  on cuda:0


## 3. Subconjunto de tuning (40% del train)

**Decision:** Crear un subconjunto estratificado del 40% del train para usar durante la búsqueda de hiperparámetros con Optuna.

**Why:** Usar el train completo en cada trial sería costoso. El 40% (~4,800 muestras) es suficiente para estimar la capacidad del modelo y reduce el tiempo por trial, manteniendo el balance de clases.

**Expected:** `X_tune` con ~4,799 muestras, misma proporción de clases que el train completo.

In [11]:
X_tune_np, _, y_tune_np, _ = train_test_split(
    X_train, y_train,
    test_size=0.6,
    stratify=y_train,
    random_state=RANDOM_STATE
)

X_tune_t, y_tune_t = to_tensor(X_tune_np, y_tune_np, DEVICE)

print(f"Tuning set: {X_tune_t.shape}  |  label balance: {y_tune_np.mean():.3f}")

Tuning set: torch.Size([4798, 13])  |  label balance: 0.500


## 4. Definición de arquitecturas

### 4A. Arquitectura MLP dinámica

**Decision:** Implementar `MLP` con constructor dual: acepta un `optuna.Trial` durante la búsqueda o un `dict` de parámetros para reentrenamiento y reconstrucción desde checkpoint.

**Why:** El constructor dual evita duplicar la lógica de construcción de capas. La capa de salida no incluye `Sigmoid` porque se usa `BCEWithLogitsLoss`, que combina sigmoid + BCE internamente para mayor estabilidad numérica.

**Expected:** Red `Linear(13,128) → ReLU → [n_layers capas ocultas] → Linear(last_hidden,1)` sin activación en la salida.

In [12]:
class MLP(nn.Module):
    """MLP con arquitectura controlada por Optuna o por un dict de parámetros."""

    def __init__(self, input_dim: int, trial_or_params):
        super().__init__()

        if isinstance(trial_or_params, optuna.Trial):
            trial = trial_or_params
            n_layers  = trial.suggest_int("n_layers", 1, 4)
            activation_name = trial.suggest_categorical("activation", ["relu", "tanh", "leakyrelu"])
            dropout_rate    = trial.suggest_float("dropout", 0.0, 0.5)
            units = [trial.suggest_int(f"n_units_l{i}", 32, 256) for i in range(n_layers)]
        else:
            params = trial_or_params
            n_layers        = params["n_layers"]
            activation_name = params["activation"]
            dropout_rate    = params["dropout"]
            units = [params[f"n_units_l{i}"] for i in range(n_layers)]

        layers = [
            nn.Linear(input_dim, FIXED_HIDDEN),
            nn.ReLU(),
        ]

        in_features = FIXED_HIDDEN
        for out_features in units:
            layers.append(nn.Linear(in_features, out_features))
            layers.append(_get_activation(activation_name))
            layers.append(nn.Dropout(dropout_rate))
            in_features = out_features

        layers.append(nn.Linear(in_features, 1))  # salida: logits (sin sigmoid)
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def _get_activation(name: str) -> nn.Module:
    return {"relu": nn.ReLU(), "tanh": nn.Tanh(), "leakyrelu": nn.LeakyReLU()}[name]


print("MLP definido OK")

MLP definido OK


### 4B. Arquitectura Residual MLP

**Decision:** Implementar `ResidualBlock` con conexión skip; si las dimensiones de entrada y salida difieren, se agrega una proyección lineal en el camino residual.

**Why:** Las conexiones residuales mitigan el problema del gradiente desvaneciente en redes más profundas y permiten que el modelo aprenda perturbaciones sobre la identidad, lo que puede acelerar la convergencia.

**Expected:** `ResidualMLP` con la misma interfaz que `MLP` (constructor dual), misma capa fija `13→128` y misma salida sin sigmoid.

In [13]:
class ResidualBlock(nn.Module):
    """Bloque residual: dos capas lineales con skip connection."""

    def __init__(self, in_dim: int, out_dim: int, activation: nn.Module, dropout: float):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            activation,
            nn.Dropout(dropout),
            nn.Linear(out_dim, out_dim),
            activation,
        )
        # proyección del camino residual solo cuando dims difieren
        self.skip = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)


class ResidualMLP(nn.Module):
    """MLP con bloques residuales; constructor dual (Trial o dict)."""

    def __init__(self, input_dim: int, trial_or_params):
        super().__init__()

        if isinstance(trial_or_params, optuna.Trial):
            trial = trial_or_params
            n_layers        = trial.suggest_int("n_layers", 1, 4)
            activation_name = trial.suggest_categorical("activation", ["relu", "tanh", "leakyrelu"])
            dropout_rate    = trial.suggest_float("dropout", 0.0, 0.5)
            units = [trial.suggest_int(f"n_units_l{i}", 32, 256) for i in range(n_layers)]
        else:
            params = trial_or_params
            n_layers        = params["n_layers"]
            activation_name = params["activation"]
            dropout_rate    = params["dropout"]
            units = [params[f"n_units_l{i}"] for i in range(n_layers)]

        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, FIXED_HIDDEN),
            nn.ReLU(),
        )

        blocks = []
        in_features = FIXED_HIDDEN
        for out_features in units:
            blocks.append(ResidualBlock(in_features, out_features,
                                        _get_activation(activation_name), dropout_rate))
            in_features = out_features

        self.blocks  = nn.Sequential(*blocks)
        self.output  = nn.Linear(in_features, 1)  # logits (sin sigmoid)

    def forward(self, x):
        x = self.input_layer(x)
        x = self.blocks(x)
        return self.output(x)


print("ResidualMLP definido OK")

ResidualMLP definido OK


### 4C. Factory de modelos

**Decision:** Centralizar la construcción de modelos en `build_model(arch, params)` para que la fase de carga de checkpoints no necesite duplicar lógica.

**Why:** Mantener un único punto de entrada para instanciar modelos facilita el uso en inferencia con nuevos datasets.

**Expected:** `build_model("mlp", params)` y `build_model("residual_mlp", params)` retornan una instancia sin inicialización de pesos (se cargarán del checkpoint).

In [14]:
def build_model(arch: str, params: dict, input_dim: int = INPUT_DIM) -> nn.Module:
    """
    Reconstruye un modelo desde un dict de parámetros.
    arch: 'mlp' | 'residual_mlp'
    """
    if arch == "mlp":
        return MLP(input_dim, params).to(DEVICE)
    elif arch == "residual_mlp":
        return ResidualMLP(input_dim, params).to(DEVICE)
    else:
        raise ValueError(f"Arquitectura desconocida: {arch}")

print("build_model definido OK")

build_model definido OK


## 5. Utilidades de entrenamiento y métricas

**Decision:** Encapsular train, evaluación y medición en funciones puras y reutilizables.

**Why:** Las 4 configuraciones de Optuna y el reentrenamiento final comparten la misma lógica; extraerla evita duplicación y bugs difíciles de rastrear.

**Expected:** Funciones importables sin estado global; cada función retorna valores numéricos o dicts.

In [15]:
criterion = nn.BCEWithLogitsLoss()


def get_optimizer(model: nn.Module, name: str, lr: float) -> optim.Optimizer:
    return {"adam": optim.Adam, "adamw": optim.AdamW}[name](model.parameters(), lr=lr)


def train_epoch(model: nn.Module, X: torch.Tensor, y: torch.Tensor,
                optimizer: optim.Optimizer, batch_size: int) -> float:
    """Entrena un epoch completo y retorna la pérdida promedio."""
    model.train()
    perm = torch.randperm(X.size(0))
    total_loss = 0.0
    n_batches  = 0
    for i in range(0, X.size(0), batch_size):
        idx     = perm[i:i + batch_size]
        batch_X = X[idx]
        batch_y = y[idx]
        optimizer.zero_grad()
        logits = model(batch_X)
        loss   = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1
    return total_loss / n_batches


def evaluate(model: nn.Module, X: torch.Tensor, y: torch.Tensor) -> dict:
    """Retorna F1 y accuracy sobre el conjunto dado."""
    model.eval()
    with torch.no_grad():
        logits = model(X)
        probs  = torch.sigmoid(logits)
        preds  = (probs > 0.5).long().cpu().numpy()
        y_true = y.long().cpu().numpy()
    return {
        "f1":       f1_score(y_true, preds, zero_division=0),
        "accuracy": accuracy_score(y_true, preds),
    }


def measure_inference_time(model: nn.Module, X: torch.Tensor, n_repeats: int = 5) -> float:
    """Retorna tiempo de inferencia promedio por muestra (segundos)."""
    model.eval()
    # Calentamiento GPU para evitar overhead de JIT en la primera pasada
    with torch.no_grad():
        _ = model(X[:1])
    torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(n_repeats):
            start = time.perf_counter()
            _ = model(X)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - start) / X.size(0))
    return float(np.mean(times))


def measure_model_size_mb(model: nn.Module) -> float:
    """Guarda el state_dict en un archivo temporal y retorna su tamaño en MB."""
    tmp = MODELS_DIR / "_tmp_size.pt"
    torch.save(model.state_dict(), tmp)
    size_mb = tmp.stat().st_size / (1024 ** 2)
    tmp.unlink()
    return size_mb


print("Utilidades definidas OK")

Utilidades definidas OK


## 6. Funciones objetivo para Optuna

### 6A. Objetivos TPE s3 inference, f1 y size

**Decision:** Una sola función objetivo por arquitectura que maximiza el F1-score en validación.

**Why:** TPE (`Tree-structured Parzen Estimator`) es un algoritmo Bayesiano que modela la distribución de hiperparámetros prometedores. Al optimizar un solo objetivo (F1), la señal de aprendizaje es clara y la convergencia suele ser más rápida que en multi-objetivo.

**Expected:** Cada trial retorna un escalar (F1). Métricas adicionales (accuracy, tiempos, tamaño) se guardan como `user_attrs` para análisis posterior.

In [20]:
def _run_trial(model_cls, trial: optuna.Trial):
    """Lógica compartida para los trials de Optuna (TPE y NSGA-II)."""
    lr         = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    opt_name   = trial.suggest_categorical("optimizer", ["adam", "adamw"])

    model     = model_cls(INPUT_DIM, trial).to(DEVICE)
    optimizer = get_optimizer(model, opt_name, lr)

    t_start = time.perf_counter()
    for _ in range(TUNING_EPOCHS):
        train_epoch(model, X_tune_t, y_tune_t, optimizer, batch_size)
    train_time = time.perf_counter() - t_start

    metrics        = evaluate(model, X_val_t, y_val_t)
    inference_time = measure_inference_time(model, X_val_t)
    model_size     = measure_model_size_mb(model)

    trial.set_user_attr("accuracy",       metrics["accuracy"])
    trial.set_user_attr("train_time_s",   train_time)
    trial.set_user_attr("inference_time", inference_time)
    trial.set_user_attr("model_size_mb",  model_size)

    return metrics["f1"], inference_time, model_size


def objective_mlp_tpe(trial):
    return _run_trial(MLP, trial)


def objective_residual_mlp_tpe(trial):
    return _run_trial(ResidualMLP, trial)


print("Objetivos TPE definidos OK")

Objetivos TPE definidos OK


### 6B. Objetivos NSGA-II (multi-objetivo)

**Decision:** Retornar una tupla `(f1, inference_time, model_size)` con direcciones `(maximize, minimize, minimize)`.

**Why:** NSGA-II explora simultáneamente el trade-off entre rendimiento predictivo y eficiencia. El resultado es una frontera de Pareto de soluciones no dominadas, permitiendo elegir la solución más adecuada para el contexto de despliegue.

**Expected:** `study.best_trials` contendrá la frontera de Pareto al finalizar los `N_TRIALS` trials.

In [21]:
def objective_mlp_nsga2(trial: optuna.Trial):
    return _run_trial(MLP, trial)  # (f1, inference_time, model_size)


def objective_residual_nsga2(trial: optuna.Trial):
    return _run_trial(ResidualMLP, trial)


print("Objetivos NSGA-II definidos OK")

Objetivos NSGA-II definidos OK


## 7. Optimización con Optuna — 4 estudios

Los 4 estudios se ejecutan secuencialmente. Cada uno genera sus trials y persiste resultados en `user_attrs`.

> **Nota de tiempo:** Con `N_TRIALS=30` y `TUNING_EPOCHS=10`, estimar ~2-5 min por estudio en CPU (varía con la arquitectura y `batch_size`). En GPU el tiempo será significativamente menor.

---

### 7.1 Config 1 — MLP + TPE

**Decision:** TPE single-objective, maximizar F1-score en validación.

**Why:** Establece una línea base sólida para el MLP usando el sampler por defecto de Optuna, que es eficiente en espacios de búsqueda continuos/categóricos mixtos.

**Expected:** `study_mlp_tpe.best_trial` con el mayor F1 encontrado entre los `N_TRIALS` trials.

In [24]:
study_mlp_tpe = optuna.create_study(
    directions=["maximize", "minimize", "minimize"],
    sampler=TPESampler(seed=RANDOM_STATE),
    study_name="mlp_tpe"
)
study_mlp_tpe.optimize(objective_mlp_tpe, n_trials=N_TRIALS, show_progress_bar=True)


100%|██████████| 30/30 [02:48<00:00,  5.62s/it]


In [25]:
pareto_c1 = study_mlp_tpe.best_trials

In [28]:
print(f"[MLP + TPE] Soluciones Pareto: {len(pareto_c1)}")
for i, t in enumerate(pareto_c1):
    print(f"  Solución {i+1}: F1={t.values[0]:.4f} | InfTime={t.values[1]:.2e}s | Size={t.values[2]:.3f}MB")

[MLP + TPE] Soluciones Pareto: 9
  Solución 1: F1=0.8724 | InfTime=5.34e-06s | Size=0.126MB
  Solución 2: F1=0.8679 | InfTime=3.23e-06s | Size=0.082MB
  Solución 3: F1=0.8391 | InfTime=1.50e-06s | Size=0.026MB
  Solución 4: F1=0.8644 | InfTime=3.16e-06s | Size=0.082MB
  Solución 5: F1=0.8487 | InfTime=1.35e-06s | Size=0.034MB
  Solución 6: F1=0.8438 | InfTime=1.16e-06s | Size=0.035MB
  Solución 7: F1=0.8641 | InfTime=3.15e-06s | Size=0.075MB
  Solución 8: F1=0.8627 | InfTime=1.22e-06s | Size=0.034MB
  Solución 9: F1=0.8744 | InfTime=5.89e-06s | Size=0.127MB


### 7.2 Config 2 — MLP + NSGA-II

**Decision:** NSGA-II multi-objetivo: maximizar F1, minimizar tiempo de inferencia y tamaño del modelo.

**Why:** Permite explorar modelos MLP que sean tanto precisos como eficientes en recursos, lo cual es relevante para despliegue en entornos con restricciones de latencia o memoria.

**Expected:** `study_mlp_nsga2.best_trials` contendrá la frontera de Pareto (soluciones no dominadas).

In [26]:
study_mlp_nsga2 = optuna.create_study(
    directions=["maximize", "minimize", "minimize"],
    sampler=NSGAIISampler(seed=RANDOM_STATE),
    study_name="mlp_nsga2"
)
study_mlp_nsga2.optimize(objective_mlp_nsga2, n_trials=N_TRIALS, show_progress_bar=True)

pareto_c2 = study_mlp_nsga2.best_trials
print(f"[MLP + NSGA-II] Soluciones Pareto: {len(pareto_c2)}")
for i, t in enumerate(pareto_c2):
    print(f"  Solución {i+1}: F1={t.values[0]:.4f} | InfTime={t.values[1]:.2e}s | Size={t.values[2]:.3f}MB")

100%|██████████| 30/30 [01:54<00:00,  3.83s/it]

[MLP + NSGA-II] Soluciones Pareto: 6
  Solución 1: F1=0.8639 | InfTime=5.29e-06s | Size=0.126MB
  Solución 2: F1=0.8615 | InfTime=3.15e-06s | Size=0.059MB
  Solución 3: F1=0.8602 | InfTime=1.34e-06s | Size=0.065MB
  Solución 4: F1=0.8354 | InfTime=1.45e-06s | Size=0.026MB
  Solución 5: F1=0.8654 | InfTime=7.62e-06s | Size=0.216MB
  Solución 6: F1=0.8687 | InfTime=8.41e-06s | Size=0.212MB


### 7.3 Config 3 — Residual MLP + TPE

**Decision:** TPE single-objective aplicado a la arquitectura con bloques residuales.

**Why:** Comparar TPE entre MLP y Residual MLP permite aislar el impacto de las skip connections bajo la misma estrategia de búsqueda.

**Expected:** `study_residual_tpe.best_trial` con el mayor F1 para Residual MLP.

In [30]:
study_residual_tpe = optuna.create_study(
    directions=["maximize", "minimize", "minimize"],
    sampler=TPESampler(seed=RANDOM_STATE),
    study_name="residual_mlp_tpe"
)
study_residual_tpe.optimize(objective_residual_mlp_tpe, n_trials=N_TRIALS, show_progress_bar=True)

pareto_c3 = study_residual_tpe.best_trials
print(f"[residualMLP + TPE] Soluciones Pareto: {len(pareto_c3)}")
for i, t in enumerate(pareto_c3):
    print(f"  Solución {i+1}: F1={t.values[0]:.4f} | InfTime={t.values[1]:.2e}s | Size={t.values[2]:.3f}MB")

100%|██████████| 30/30 [04:19<00:00,  8.66s/it]

[residualMLP + TPE] Soluciones Pareto: 8
  Solución 1: F1=0.8643 | InfTime=7.35e-06s | Size=0.237MB
  Solución 2: F1=0.8367 | InfTime=2.23e-06s | Size=0.048MB
  Solución 3: F1=0.8724 | InfTime=2.05e-05s | Size=0.537MB
  Solución 4: F1=0.8546 | InfTime=2.39e-06s | Size=0.058MB
  Solución 5: F1=0.8642 | InfTime=5.51e-06s | Size=0.125MB
  Solución 6: F1=0.8606 | InfTime=2.58e-06s | Size=0.070MB
  Solución 7: F1=0.8338 | InfTime=2.10e-06s | Size=0.047MB
  Solución 8: F1=0.8678 | InfTime=1.31e-05s | Size=0.373MB


### 7.4 Config 4 — Residual MLP + NSGA-II

**Decision:** NSGA-II multi-objetivo aplicado a Residual MLP.

**Why:** Cierra la matriz 2×2 (arquitectura × sampler), permitiendo la comparación completa entre las 4 configuraciones en la Fase 12.

**Expected:** Frontera de Pareto para Residual MLP.

In [32]:
study_residual_nsga2 = optuna.create_study(
    directions=["maximize", "minimize", "minimize"],
    sampler=NSGAIISampler(seed=RANDOM_STATE),
    study_name="residual_mlp_nsga2"
)
study_residual_nsga2.optimize(objective_residual_nsga2, n_trials=N_TRIALS, show_progress_bar=True)

pareto_c4 = study_residual_nsga2.best_trials
print(f"[ResidualMLP + NSGA-II] Soluciones Pareto: {len(pareto_c4)}")
for i, t in enumerate(pareto_c4):
    print(f"  Solución {i+1}: F1={t.values[0]:.4f} | InfTime={t.values[1]:.2e}s | Size={t.values[2]:.3f}MB")

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 30/30 [04:50<00:00,  9.67s/it]

[ResidualMLP + NSGA-II] Soluciones Pareto: 3
  Solución 1: F1=0.8570 | InfTime=6.13e-06s | Size=0.148MB
  Solución 2: F1=0.8666 | InfTime=2.65e-06s | Size=0.170MB
  Solución 3: F1=0.8484 | InfTime=2.25e-06s | Size=0.048MB


## 8. Selección de mejores hiperparámetros

**Decision:** Para TPE, seleccionar `study.best_trial` (máximo F1). Para NSGA-II, usar la solución Pareto con mayor F1 como default; el criterio es configurable vía `PARETO_SELECTION`.

**Why:** Una selección automática reproducible es necesaria para el reentrenamiento final. Sin embargo, documentar el criterio permite ajustarlo sin modificar el código de entrenamiento.

**Expected:** Un dict `best_params` por configuración, listo para `build_model(arch, params)`.

In [40]:
pareto_c1 = select_pareto(study_mlp_tpe.best_trials)
pareto_c2 = select_pareto(study_mlp_nsga2.best_trials)
pareto_c3 = select_pareto(study_residual_tpe.best_trials)
pareto_c4 = select_pareto(study_residual_nsga2.best_trials)

In [41]:
print(pareto_c2.params)

{'learning_rate': 0.0005151638228695856, 'batch_size': 32, 'optimizer': 'adam', 'n_layers': 4, 'activation': 'relu', 'dropout': 0.24580793755841618, 'n_units_l0': 138, 'n_units_l1': 70, 'n_units_l2': 129, 'n_units_l3': 121}


In [42]:
# ── Criterio de selección Pareto ─────────────────────────────────────────────
# "best_f1"    : solución con mayor F1 en la frontera de Pareto
# "balanced"   : normalizar F1, tiempo e inverso-tamaño y elegir la suma mayor
PARETO_SELECTION = "best"

def select_pareto_f1(trials, method: str = PARETO_SELECTION):
    """Retorna el trial Pareto seleccionado según el criterio indicado."""
    if method == "best":
        return max(trials, key=lambda t: t.values[0])
    elif method == "balanced":
        f1s    = np.array([t.values[0] for t in trials])
        times  = np.array([t.values[1] for t in trials])
        sizes  = np.array([t.values[2] for t in trials])
        norm   = (f1s / f1s.max()) - (times / times.max()) - (sizes / sizes.max())
        return trials[int(np.argmax(norm))]
    else:
        raise ValueError(f"Criterio desconocido: {method}")
    
def select_pareto_inference(trials, method: str = PARETO_SELECTION):
    """Retorna el trial Pareto seleccionado según el criterio indicado."""
    if method == "best":
        return min(trials, key=lambda t: t.values[1])
    elif method == "balanced":
        f1s    = np.array([t.values[0] for t in trials])
        times  = np.array([t.values[1] for t in trials])
        sizes  = np.array([t.values[2] for t in trials])
        norm   = (f1s / f1s.min()) - (times / times.min()) - (sizes / sizes.min())
        return trials[int(np.argmin(norm))]
    else:
        raise ValueError(f"Criterio desconocido: {method}")
def select_pareto_size(trials, method: str = PARETO_SELECTION):
    """Retorna el trial Pareto seleccionado según el criterio indicado."""
    if method == "best":
        return min(trials, key=lambda t: t.values[2])
    elif method == "balanced":
        f1s    = np.array([t.values[0] for t in trials])
        times  = np.array([t.values[1] for t in trials])
        sizes  = np.array([t.values[2] for t in trials])
        norm   = (f1s / f1s.min()) - (times / times.min()) - (sizes / sizes.min())
        return trials[int(np.argmin(norm))]
    else:
        raise ValueError(f"Criterio desconocido: {method}")


# ── Extraer mejores parámetros basados en maximizar f1 ────────────────────────────────────────────────
best_params_mlp_tpe_f1       = select_pareto_f1(study_mlp_tpe.best_trials)
best_params_mlp_nsga2_f1     = select_pareto_f1(study_mlp_nsga2.best_trials)
best_params_residual_tpe_f1  = select_pareto_f1(study_residual_tpe.best_trials)
best_params_residual_nsga2_f1 = select_pareto_f1(study_residual_nsga2.best_trials)


# ── Extraer mejores parámetros basados minimiza inference time  ────────────────────────────────────────────────
best_params_mlp_tpe_inference       = select_pareto_inference(study_mlp_tpe.best_trials)
best_params_mlp_nsga2_inference     = select_pareto_inference(study_mlp_nsga2.best_trials)
best_params_residual_tpe_inference  = select_pareto_inference(study_residual_tpe.best_trials)
best_params_residual_nsga2_inference = select_pareto_inference(study_residual_nsga2.best_trials)

# ── Extraer mejores parámetros basados en minimizar tama;o del modelo  ────────────────────────────────────────────────
best_params_mlp_tpe_size       = select_pareto_size(study_mlp_tpe.best_trials)
best_params_mlp_nsga2_size     = select_pareto_size(study_mlp_nsga2.best_trials)
best_params_residual_tpe_size  = select_pareto_size(study_residual_tpe.best_trials)
best_params_residual_nsga2_size = select_pareto_size(study_residual_nsga2.best_trials)




configs_f1 = [
    ("mlp", "tpe",
     best_params_mlp_tpe_f1.params,
     best_params_mlp_tpe_f1.values[0]),

    ("mlp", "nsga2",
     best_params_mlp_nsga2_f1.params,
     best_params_mlp_nsga2_f1.values[0]),

    ("residual_mlp", "tpe",
     best_params_residual_tpe_f1.params,
     best_params_residual_tpe_f1.values[0]),

    ("residual_mlp", "nsga2",
     best_params_residual_nsga2_f1.params,
     best_params_residual_nsga2_f1.values[0]),
]

configs_inference = [
    ("mlp", "tpe",
     best_params_mlp_tpe_inference.params,
     best_params_mlp_tpe_inference.values[0]),

    ("mlp", "nsga2",
     best_params_mlp_nsga2_inference.params,
     best_params_mlp_nsga2_inference.values[0]),

    ("residual_mlp", "tpe",
     best_params_residual_tpe_inference.params,
     best_params_residual_tpe_inference.values[0]),

    ("residual_mlp", "nsga2",
     best_params_residual_nsga2_inference.params,
     best_params_residual_nsga2_inference.values[0]),
]

configs_size = [
    ("mlp", "tpe",
     best_params_mlp_tpe_size.params,
     best_params_mlp_tpe_size.values[0]),

    ("mlp", "nsga2",
     best_params_mlp_nsga2_size.params,
     best_params_mlp_nsga2_size.values[0]),

    ("residual_mlp", "tpe",
     best_params_residual_tpe_size.params,
     best_params_residual_tpe_size.values[0]),

    ("residual_mlp", "nsga2",
     best_params_residual_nsga2_size.params,
     best_params_residual_nsga2_size.values[0]),
]



In [49]:
def print_selected_solution(name, trial, pareto_trials):

    solution_number = pareto_trials.index(trial) + 1

    params = trial.params

    units = [
        str(params[f"n_units_l{i}"])
        for i in range(params["n_layers"])
    ]

    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    print(f"Pareto Solution : {solution_number}")
    print(f"Trial Number    : {trial.number}")

    print("\nMetrics")
    print("-" * 40)
    print(f"F1 Score        : {trial.values[0]:.4f}")
    print(f"Inference Time  : {trial.values[1]*1e6:.2f} μs")
    print(f"Model Size      : {trial.values[2]:.4f} MB")
    print(f"Accuracy        : {trial.user_attrs['accuracy']:.4f}")

    print("\nArchitecture")
    print("-" * 40)
    print(f"Layers          : {params['n_layers']}")
    print(f"Activation      : {params['activation']}")
    print(f"Dropout         : {params['dropout']:.4f}")
    print(f"Optimizer       : {params['optimizer']}")
    print(f"Batch Size      : {params['batch_size']}")
    print(f"Learning Rate   : {params['learning_rate']:.6f}")
    print(f"Units           : {' - '.join(units)}")


# ============================================================
# BEST F1
# ============================================================

print("\n\n########## BEST F1 ##########")

print_selected_solution(
    "MLP + TPE",
    best_params_mlp_tpe_f1,
    study_mlp_tpe.best_trials
)

print_selected_solution(
    "MLP + NSGA-II",
    best_params_mlp_nsga2_f1,
    study_mlp_nsga2.best_trials
)

print_selected_solution(
    "ResidualMLP + TPE",
    best_params_residual_tpe_f1,
    study_residual_tpe.best_trials
)

print_selected_solution(
    "ResidualMLP + NSGA-II",
    best_params_residual_nsga2_f1,
    study_residual_nsga2.best_trials
)


# ============================================================
# BEST INFERENCE TIME
# ============================================================

print("\n\n########## BEST INFERENCE ##########")

print_selected_solution(
    "MLP + TPE",
    best_params_mlp_tpe_inference,
    study_mlp_tpe.best_trials
)

print_selected_solution(
    "MLP + NSGA-II",
    best_params_mlp_nsga2_inference,
    study_mlp_nsga2.best_trials
)

print_selected_solution(
    "ResidualMLP + TPE",
    best_params_residual_tpe_inference,
    study_residual_tpe.best_trials
)

print_selected_solution(
    "ResidualMLP + NSGA-II",
    best_params_residual_nsga2_inference,
    study_residual_nsga2.best_trials
)


# ============================================================
# BEST MODEL SIZE
# ============================================================

print("\n\n########## BEST MODEL SIZE ##########")

print_selected_solution(
    "MLP + TPE",
    best_params_mlp_tpe_size,
    study_mlp_tpe.best_trials
)

print_selected_solution(
    "MLP + NSGA-II",
    best_params_mlp_nsga2_size,
    study_mlp_nsga2.best_trials
)

print_selected_solution(
    "ResidualMLP + TPE",
    best_params_residual_tpe_size,
    study_residual_tpe.best_trials
)

print_selected_solution(
    "ResidualMLP + NSGA-II",
    best_params_residual_nsga2_size,
    study_residual_nsga2.best_trials
)



########## BEST F1 ##########

MLP + TPE
Pareto Solution : 9
Trial Number    : 29

Metrics
----------------------------------------
F1 Score        : 0.8744
Inference Time  : 5.89 μs
Model Size      : 0.1273 MB
Accuracy        : 0.8708

Architecture
----------------------------------------
Layers          : 1
Activation      : relu
Dropout         : 0.0597
Optimizer       : adam
Batch Size      : 64
Learning Rate   : 0.005448
Units           : 237

MLP + NSGA-II
Pareto Solution : 6
Trial Number    : 25

Metrics
----------------------------------------
F1 Score        : 0.8687
Inference Time  : 8.41 μs
Model Size      : 0.2118 MB
Accuracy        : 0.8678

Architecture
----------------------------------------
Layers          : 4
Activation      : relu
Dropout         : 0.2458
Optimizer       : adam
Batch Size      : 32
Learning Rate   : 0.000515
Units           : 138 - 70 - 129 - 121

ResidualMLP + TPE
Pareto Solution : 3
Trial Number    : 18

Metrics
----------------------------------

In [75]:
#GUARDAR LAS MEJORES CONFIGURACIONES:
import json

best_configs = [
    {
        "architecture": "mlp",
        "sampler": "tpe",
        "params": best_params_mlp_tpe_f1.params,  # tu diccionario
        "f1": 0.8744,
        "inference_us": 5.89,
        "size_mb": 0.1273,
        "trial": 29,
        "pareto_solution": 9
    },
    {
        "architecture": "residual_mlp",
        "sampler": "nsga2",
        "params": best_params_residual_nsga2_inference.params,
        "f1": 0.8666,
        "inference_us": 2.65,
        "size_mb": 0.1701,
        "trial": 4,
        "pareto_solution": 2
    },
    {
        "architecture": "mlp",
        "sampler": "nsga2",
        "params": best_params_mlp_nsga2_inference.params,
        "f1": 0.8602,
        "inference_us": 1.34,
        "size_mb": 0.0654,
        "trial": 4,
        "pareto_solution": 3
    }
]

with open("best_configs.json", "w") as f:
    json.dump(best_configs, f, indent=2)

## 10. Guardado de modelos

**Decision:** Persistir cada modelo final como un checkpoint `.pt` que incluye `state_dict`, parámetros Optuna, metadatos de arquitectura y una referencia al scaler.

**Why:** Guardar todo en un único archivo por modelo permite cargarlo en un notebook de inferencia o en un dataset diferente sin necesidad de re-ejecutar Optuna ni conocer la arquitectura de antemano. La referencia al scaler facilita el preprocesamiento de nuevos datos de forma consistente.

**Expected:** 4 archivos `.pt` en `models/gpu` más un `experiments_summary.json`.

### 10.1 Helper de carga para inferencia

**Decision:** Definir `load_model(checkpoint_path, device)` como utilidad reutilizable.

**Why:** Esta función permite usar cualquiera de los 4 modelos en inferencia sobre un dataset diferente sin importar en qué dispositivo fue entrenado originalmente.

**Expected:** Retorna un `nn.Module` en modo `eval()` listo para inferencia.

## 11. Evaluación final en Test

**Decision:** Evaluar los 4 modelos guardados sobre el conjunto de test para obtener métricas finales no sesgadas.

**Why:** El test set no fue visto durante la búsqueda de hiperparámetros ni durante el reentrenamiento. Las métricas reportadas aquí son las que se usarán en la comparación entre notebooks CPU/GPU y en el paper.

**Expected:** F1, accuracy, tiempo de inferencia y tamaño de modelo para cada configuración; tabla resumen impresa.

## 12. Comparación de resultados y exportación

**Decision:** Compilar todos los resultados en un DataFrame, exportar a CSV y a `experiments_summary.json` para fusión posterior con el notebook del otro dispositivo.

**Why:** El CSV permite combinar los resultados de CPU y GPU en un notebook separado (o en el paper) haciendo `pd.concat([df_cpu, df_gpu])`. El JSON facilita la lectura programática.

**Expected:** `results/results_gpu.csv` y `results/experiments_summary_gpu.json` creados con las 4 configuraciones evaluadas en `GPU (CUDA)`.

### 12.1 Comparación cross-device

**Decision:** Documentar cómo fusionar los resultados de ambos notebooks para la comparación final CPU vs GPU.

**Why:** Los dos notebooks generan archivos independientes; la fusión es el último paso del flujo experimental.

**Expected:** Las instrucciones permiten construir la tabla comparativa completa (8 filas: 4 configs × 2 devices).

```python
# En un notebook de análisis o en el paper:
import pandas as pd

df_cpu = pd.read_csv("traning/results/results_cpu.csv")
df_gpu = pd.read_csv("traning/results/results_gpu.csv")
df_all = pd.concat([df_cpu, df_gpu], ignore_index=True)

# Tabla pivote para comparación directa
pivot = df_all.pivot_table(
    index=["arch", "sampler"],
    columns="device",
    values=["f1_test", "accuracy_test", "inference_time", "model_size_mb"]
)
print(pivot)
```